# Vector to Raster: `Rasterizer`

The `Rasterizer` processor converts vector data (shapefiles, GeoPackages,
GeoDataFrames) into single-band rasters aligned to a `SpatialSpec`.
It wraps `rasterio.features.rasterize` with caching.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import geopandas as gpd

from pygeodata import DataLoader, SpatialSpec, load, setconfig
from pygeodata.processors.rasterizer import Rasterizer
from pyproj import CRS
from affine import Affine

setconfig(path_data_processed=Path("./data/processed"))

spec = SpatialSpec(
    crs=CRS.from_epsg(4326),
    transform=Affine(0.5, 0, -180, 0, -0.5, 90),
    shape=(360, 720),
)

## 1. Burn a constant value — binary presence mask

In [ ]:
@dataclass
class CountryMaskLoader(DataLoader):
    """Rasterize country polygons to a presence/absence mask."""
    src: Path = Path("data/raw/countries.gpkg")

    @property
    def processor(self):
        return Rasterizer(
            srcpath=self.src,
            values=1,           # all polygons get value 1
            dtype=np.uint8,
            fill_value=0,
            all_touched=True,   # burn all pixels that touch the polygon
        )


mask_loader = CountryMaskLoader()
print(mask_loader)
print("Output path:", mask_loader.get_processed_path(spec))

## 2. Burn a numeric attribute column

In [ ]:
@dataclass
class PopulationDensityLoader(DataLoader):
    """Rasterize admin-unit population density attribute."""
    src: Path = Path("data/raw/admin_units.gpkg")

    @property
    def processor(self):
        return Rasterizer(
            srcpath=self.src,
            values="pop_density",   # column name to burn into pixels
            dtype=np.float32,
            fill_value=np.nan,
            nodata_value=np.nan,
        )

## 3. Burn the dataframe index

`values='index'` burns the integer row-index of the GeoDataFrame.
Useful for lookup rasters that map pixels back to vector features.

In [ ]:
@dataclass
class BiomeIndexLoader(DataLoader):
    src: Path = Path("data/raw/biomes.gpkg")

    @property
    def processor(self):
        return Rasterizer(
            srcpath=self.src,
            values="index",   # 0-based integer row index
            dtype=np.int16,
            fill_value=-1,
            nodata_value=-1,
        )

## 4. Dynamic vector loading via `load_df`

Supply a callable instead of a file path. The callable receives the
`SpatialSpec` and returns a `GeoDataFrame`, allowing on-the-fly
filtering based on the target bounds.

In [ ]:
def load_clipped_rivers(spec: SpatialSpec) -> gpd.GeoDataFrame:
    gdf = gpd.read_file("data/raw/rivers.gpkg").to_crs(spec.crs)
    b = spec.bounds
    return gdf.cx[b.left:b.right, b.bottom:b.top]


@dataclass
class RiverLoader(DataLoader):
    @property
    def processor(self):
        return Rasterizer(
            load_df=load_clipped_rivers,
            values=1,
            dtype=np.uint8,
            fill_value=0,
        )


river_loader = RiverLoader()
# river_mask = load(river_loader, spec)  # uncomment with real data
print(river_loader)